In [ ]:
# Install required packages and load environment variables
%pip install anthropic
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()
client = Anthropic()
model = "claude-sonnet-4-6"

In [ ]:
# Helper functions â€” chat() now supports stop_sequences
def add_user_message(messages, text):
    messages.append({"role": "user", "content": text})

def add_assistant_message(messages, text):
    messages.append({"role": "assistant", "content": text})

def chat(messages, system=None, temperature=1.0, stop_sequences=None):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
    }
    if system:
        params["system"] = system
    if stop_sequences:
        params["stop_sequences"] = stop_sequences
    message = client.messages.create(**params)
    return message.content[0].text

In [ ]:
# Tanpa teknik apapun â€” Claude jawab dengan markdown formatting
messages = []
add_user_message(messages, "Generate a very short event bridge rule as json")

text = chat(messages)
text

In [ ]:
# Teknik 1: Prefill assistant message
# CATATAN: Prefill hanya support di Claude 3, bukan Claude 4
# Ganti model ke claude-3-5-haiku-20241022 untuk cell ini
messages = []
add_user_message(messages, "Generate a very short event bridge rule as json")
add_assistant_message(messages, "```json")  # <-- prefill

text = client.messages.create(
    model="claude-3-5-haiku-20241022",  # Claude 3 support prefill
    max_tokens=1000,
    messages=messages,
).content[0].text
text

In [ ]:
# Teknik 2: Prefill + Stop Sequence (Claude 3)
messages = []
add_user_message(messages, "Generate a very short event bridge rule as json")
add_assistant_message(messages, "```json")  # <-- prefill

text = client.messages.create(
    model="claude-3-5-haiku-20241022",
    max_tokens=1000,
    messages=messages,
    stop_sequences=["```"],  # <-- stop sebelum penutup
).content[0].text
text

In [ ]:
# Alternatif untuk Claude 4: system prompt bukan prefill
# Claude 4 tidak support prefill, tapi bisa pakai system prompt
import json

messages = []
add_user_message(messages, "Generate a very short event bridge rule as json")

text = chat(
    messages,
    system="Respond with only raw valid JSON. No markdown, no explanation."
)
data = json.loads(text.strip())
print(json.dumps(data, indent=2))